# 07 - Bronze: Fragile States Index

This notebook ingests Fragile States Index JSONL produced by the local extractor and writes `bronze.fsi_raw`.

Official FSI Excel downloads currently cover 2006-2023. If the local extractor is run with `--start-year 1990 --end-year 2024`, it writes the available official years and reports the missing requested years.

Local extraction:

```bash
python3 extraction/extract/fsi_extract.py \
  --all-cemac-ecowas \
  --start-year 1990 --end-year 2024 \
  --out data/raw/fsi/cemac_ecowas_fsi_1990_2024.jsonl
```

Upload to Databricks:

```bash
databricks fs mkdirs dbfs:/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/fsi -p cemac-project
databricks fs cp data/raw/fsi/cemac_ecowas_fsi_1990_2024.jsonl \
  dbfs:/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/fsi/cemac_ecowas_fsi_1990_2024.jsonl \
  --overwrite -p cemac-project
```

In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("USE SCHEMA bronze")

VOLUME_PATH = "/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/fsi/"
loaded_at = datetime.now(timezone.utc)

print("Catalog: cemac_ecowas_aes_trade")
print("Schema: bronze")
print(f"Volume path: {VOLUME_PATH}")

In [ ]:
# Cell 2 - List uploaded JSONL files
files = [f for f in dbutils.fs.ls(VOLUME_PATH) if f.name.endswith(".jsonl")]

if not files:
    raise FileNotFoundError(
        f"No .jsonl files found at {VOLUME_PATH}. Run fsi_extract.py locally and upload the output first."
    )

print(f"Found {len(files)} JSONL file(s):")
for file in files:
    print(f"  {file.name} ({file.size:,} bytes)")

In [ ]:
# Cell 3 - Read JSONL and normalize schema
paths = [file.path for file in files]
raw_df = spark.read.json(paths)

df = raw_df.select(
    F.col("source").cast("string").alias("source"),
    F.col("dataset").cast("string").alias("dataset"),
    F.col("country_iso3").cast("string").alias("country_iso3"),
    F.col("country_name").cast("string").alias("country_name"),
    F.col("country_name_source").cast("string").alias("country_name_source"),
    F.col("year").cast("int").alias("year"),
    F.col("rank").cast("int").alias("rank"),
    F.col("indicator_code").cast("string").alias("indicator_code"),
    F.col("indicator_name").cast("string").alias("indicator_name"),
    F.col("category").cast("string").alias("category"),
    F.col("value").cast("double").alias("value"),
    F.col("source_url").cast("string").alias("source_url"),
    F.col("extracted_at").cast("string").alias("extracted_at"),
).dropDuplicates(["country_iso3", "year", "indicator_code"])

df = df.withColumn("loaded_at", F.lit(loaded_at))

print(f"Rows: {df.count():,}")
df.printSchema()
df.show(10, truncate=False)

In [ ]:
# Cell 4 - Write bronze.fsi_raw
spark.sql("DROP TABLE IF EXISTS bronze.fsi_raw")

(df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.fsi_raw"))

print("Write complete: bronze.fsi_raw")

In [ ]:
# Cell 5 - Verification and sanity checks
print("Overall coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT country_iso3) AS countries,
        COUNT(DISTINCT indicator_code) AS indicators,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        COUNT(value) AS non_null_values
    FROM bronze.fsi_raw
""").show()

print("Per-indicator coverage:")
spark.sql("""
    SELECT
        indicator_code,
        indicator_name,
        category,
        COUNT(*) AS rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        COUNT(value) AS non_null_values
    FROM bronze.fsi_raw
    GROUP BY indicator_code, indicator_name, category
    ORDER BY indicator_code
""").show(truncate=False)

print("Latest total fragility score:")
spark.sql("""
    SELECT
        country_iso3,
        country_name,
        year,
        rank,
        ROUND(value, 1) AS total_score
    FROM bronze.fsi_raw
    WHERE indicator_code = 'TOTAL'
      AND year = (SELECT MAX(year) FROM bronze.fsi_raw)
    ORDER BY value DESC
""").show(25, truncate=False)

print("CEMAC latest sub-indicator profile:")
spark.sql("""
    SELECT
        country_iso3,
        indicator_code,
        indicator_name,
        ROUND(value, 1) AS score
    FROM bronze.fsi_raw
    WHERE year = (SELECT MAX(year) FROM bronze.fsi_raw)
      AND country_iso3 IN ('CMR', 'CAF', 'TCD', 'COG', 'GNQ', 'GAB')
      AND indicator_code != 'TOTAL'
    ORDER BY country_iso3, indicator_code
""").show(100, truncate=False)